# Wan2.1 T2V + Lynx (Identity) on Free T4 – ComfyUI Colab

This notebook sets up:

- ComfyUI with GGUF support  
- Wan2.1-T2V-14B Q4_K_M GGUF (T2V, quantized for T4)  
- Kijai's WanVideoWrapper + Lynx for identity-preserving text-to-video  
- Four user LoRA slots, each with its own strength control  
- Lightx2v T2V speed LoRA for fast 4–5 step inference (T4 friendly)

It follows the same style as the Faster Wan2.1 Causvid + Lightx2v notebook, but targets text-to-video with Lynx identity control instead of image-to-video.

You will control generation from this notebook, but the actual node graph runs in ComfyUI. Once ComfyUI is up, load the wanvideo_T2V_14B_lynx_example_01.json workflow (or your modified one) and wire its UNet through the LoRA chain this notebook configures.


In [1]:
# @markdown # 1. Prepare Environment (ComfyUI + GGUF + WanVideoWrapper + Lynx)
!nvidia-smi

import os, sys, shutil, textwrap

%cd /content

!apt-get update -y
!apt-get install -y aria2

# Basic deps – pinned to versions that work well on Colab T4
!pip install -q torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu118
!pip install -q torchsde einops diffusers accelerate xformers==0.0.29.post2 triton==3.2.0 sageattention==1.0.6
!pip install -q av spandrel albumentations insightface onnx opencv-python segment_anything ultralytics onnxruntime onnxruntime-gpu

# ComfyUI core
if not os.path.exists("/content/ComfyUI"):
    !git clone https://github.com/comfyanonymous/ComfyUI /content/ComfyUI

%cd /content/ComfyUI/custom_nodes

# GGUF loader
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI_GGUF"):
    !git clone https://github.com/Isi-dev/ComfyUI_GGUF.git

# KJ nodes (optimizations etc.)
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-KJNodes"):
    !git clone https://github.com/kijai/ComfyUI-KJNodes.git

# WanVideoWrapper (Lynx + Wan video)
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-WanVideoWrapper"):
    !git clone https://github.com/kijai/ComfyUI-WanVideoWrapper.git

# ComfyUI-Manager (optional, for managing/updating nodes)
if not os.path.exists("/content/ComfyUI/custom_nodes/ComfyUI-Manager"):
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git

%cd /content/ComfyUI

# Install Python deps for custom nodes (if any extra are needed)
!pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-KJNodes/requirements.txt || echo "KJNodes extra reqs done or not needed."
!pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-WanVideoWrapper/requirements.txt || echo "WanVideoWrapper extra reqs done or not needed."

print("Environment prepared. Next: configure models and LoRAs in the next cell.")


Thu Dec 11 19:07:31 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [15]:
%cd /content/ComfyUI
!git tag -f v0.4.0 || true

/content/ComfyUI


In [2]:
# @markdown # 2. Models and LoRAs – Configure and Download

from pathlib import Path
import os
from urllib.parse import urlparse, urlunparse, parse_qsl, urlencode

# @markdown ### (Optional) CivitAI API key
# @markdown If you download models/LoRAs from CivitAI, set your API key here.
civitai_api_key = "ccff51b481b90062f9fb013caf8e1451"  # @param {type:"string"}

# @markdown ### Base model + text encoder + VAE (Wan2.1 T2V 14B GGUF)
wan_gguf_url = "https://huggingface.co/city96/Wan2.1-T2V-14B-gguf/resolve/main/wan2.1-t2v-14b-Q4_K_M.gguf"  # @param {type:"string"}
text_encoder_url = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors"  # @param {type:"string"}
vae_url = "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors"  # @param {type:"string"}

# @markdown ### Lynx layers (IP + Ref + Resampler)
lynx_ip_layers_url = "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Lynx/Wan2_1-T2V-14B-Lynx_full_ip_layers_fp16.safetensors"  # @param {type:"string"}
lynx_ref_layers_url = "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Lynx/Wan2_1-T2V-14B-Lynx_full_ref_layers_fp16.safetensors"  # @param {type:"string"}
lynx_resampler_url = "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/Lynx/lynx_full_resampler_fp32.safetensors"  # @param {type:"string"}

# @markdown ### Lightx2v T2V LoRA (speed)
lightx2v_lora_url = "https://huggingface.co/lightx2v/Wan2.1-T2V-14B-StepDistill-CfgDistill-Lightx2v/resolve/main/loras/Wan21_T2V_14B_lightx2v_cfg_step_distill_lora_rank64.safetensors"  # @param {type:"string"}

# @markdown ---
# @markdown ### 4 General LoRA slots (style / motion / whatever)
use_lora1 = True  # @param {type:"boolean"}
lora1_url = "https://civitai.com/api/download/models/2336470?type=Model&format=SafeTensor"  # @param {type:"string"}

use_lora2 = True  # @param {type:"boolean"}
lora2_url = "https://civitai.com/api/download/models/2021242?type=Model&format=SafeTensor"  # @param {type:"string"}

use_lora3 = True  # @param {type:"boolean"}
lora3_url = "https://civitai.com/api/download/models/1971163?type=Model&format=SafeTensor"  # @param {type:"string"}

use_lora4 = True  # @param {type:"boolean"}
lora4_url = "https://civitai.com/api/download/models/2022080?type=Model&format=SafeTensor"  # @param {type:"string"}

print("Configured URLs. Now downloading models and LoRAs...")

def maybe_add_civitai_token(url: str) -> str:
    """If URL points to civitai.com and a civitai_api_key is set,
    append ?token=<key> (or &token=<key>) unless already present."""
    if not url or "civitai.com" not in url.lower():
        return url
    if not civitai_api_key:
        return url
    parsed = urlparse(url)
    query = dict(parse_qsl(parsed.query))
    if "token" in query:
        return url  # already has token
    query["token"] = civitai_api_key
    new_query = urlencode(query)
    new_parsed = parsed._replace(query=new_query)
    new_url = urlunparse(new_parsed)
    return new_url

def download_file(url: str, dest_dir: str) -> str:
    if not url:
        return ""
    # Handle civitai auth if needed
    url = maybe_add_civitai_token(url)

    Path(dest_dir).mkdir(parents=True, exist_ok=True)
    filename = url.split("/")[-1].split("?")[0]
    dest_path = os.path.join(dest_dir, filename)
    if os.path.exists(dest_path):
        print(f"Already exists: {dest_path}")
        return dest_path
    print(f"Downloading {url} -> {dest_path}")
    # Use aria2c if available for faster downloads; fall back to wget
    cmd = f'aria2c --console-log-level=error -c -x 16 -s 16 -k 1M "{url}" -d "{dest_dir}" -o "{filename}"'
    rc = os.system(cmd)
    if rc != 0:
        print("aria2c failed or not installed, falling back to wget...")
        cmd2 = f'wget -O "{dest_path}" "{url}"'
        rc2 = os.system(cmd2)
        if rc2 != 0:
            raise RuntimeError(f"Download failed for {url}")
    return dest_path

# Base paths
diffusion_dir = "/content/ComfyUI/models/unet"
text_enc_dir = "/content/ComfyUI/models/text_encoders"
vae_dir = "/content/ComfyUI/models/vae"
lora_dir = "/content/ComfyUI/models/loras"
lynx_dir = "/content/ComfyUI/models/lynx"

# Download core Wan2.1 T2V GGUF and encoders
wan_gguf_path = download_file(wan_gguf_url, diffusion_dir)
text_enc_path = download_file(text_encoder_url, text_enc_dir)
vae_path = download_file(vae_url, vae_dir)

# Download Lynx layers
lynx_ip_path = download_file(lynx_ip_layers_url, lynx_dir)
lynx_ref_path = download_file(lynx_ref_layers_url, lynx_dir)
lynx_resampler_path = download_file(lynx_resampler_url, lynx_dir)

# Download Lightx2v T2V LoRA
lightx2v_lora_path = download_file(lightx2v_lora_url, lora_dir)

# Download up to 4 general LoRAs
lora_paths = []

if use_lora1 and lora1_url:
    lora_paths.append(("lora1", download_file(lora1_url, lora_dir)))
if use_lora2 and lora2_url:
    lora_paths.append(("lora2", download_file(lora2_url, lora_dir)))
if use_lora3 and lora3_url:
    lora_paths.append(("lora3", download_file(lora3_url, lora_dir)))
if use_lora4 and lora4_url:
    lora_paths.append(("lora4", download_file(lora4_url, lora_dir)))

print("\nDownloads complete.")
print("Wan GGUF:", wan_gguf_path)
print("Text encoder:", text_enc_path)
print("VAE:", vae_path)
print("Lynx:", lynx_ip_path, lynx_ref_path, lynx_resampler_path)
print("Lightx2v LoRA:", lightx2v_lora_path)
print("User LoRAs:", lora_paths)


Configured URLs. Now downloading models and LoRAs...

Downloads complete.
Wan GGUF: /content/ComfyUI/models/unet/wan2.1-t2v-14b-Q4_K_M.gguf
Text encoder: /content/ComfyUI/models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors
VAE: /content/ComfyUI/models/vae/wan_2.1_vae.safetensors
Lynx: /content/ComfyUI/models/lynx/Wan2_1-T2V-14B-Lynx_full_ip_layers_fp16.safetensors /content/ComfyUI/models/lynx/Wan2_1-T2V-14B-Lynx_full_ref_layers_fp16.safetensors /content/ComfyUI/models/lynx/lynx_full_resampler_fp32.safetensors
Lightx2v LoRA: /content/ComfyUI/models/loras/Wan21_T2V_14B_lightx2v_cfg_step_distill_lora_rank64.safetensors
User LoRAs: [('lora1', '/content/ComfyUI/models/loras/2336470'), ('lora2', '/content/ComfyUI/models/loras/2021242'), ('lora3', '/content/ComfyUI/models/loras/1971163'), ('lora4', '/content/ComfyUI/models/loras/2022080')]


In [9]:
%cd /content/ComfyUI
!git pull

/content/ComfyUI
Already up to date.


In [10]:
%cd /content/ComfyUI/custom_nodes/ComfyUI-Manager
!git pull

/content/ComfyUI/custom_nodes/ComfyUI-Manager
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 60 (delta 42), reused 43 (delta 31), pack-reused 3 (from 1)
Unpacking objects: 100% (60/60), 967.50 KiB | 1.27 MiB/s, done.
From https://github.com/ltdrdata/ComfyUI-Manager
   22acaa1d..fae909de  main       -> origin/main
Updating 22acaa1d..fae909de
Fast-forward
 custom-node-list.json                |    76 +-
 extension-node-map.json              |    91 +-
 github-stats-cache.json              | 10452 +++++++++++++++++----------------
 github-stats.json                    |  5504 ++++++++---------
 node_db/dev/extension-node-map.json  |     7 +-
 node_db/legacy/custom-node-list.json |    10 +
 node_db/new/custom-node-list.json    |   170 +-
 node_db/new/extension-node-map.json  |    91 +-
 8 files changed, 8276 insertions(+), 8125 deletions(-)


In [ ]:
# @markdown # 3. Launch ComfyUI (Lynx T2V Workflow)

import subprocess, threading, time, os, re
from IPython.display import HTML

%cd /content/ComfyUI

# @markdown ### Tunnel choice (Colab proxy vs Cloudflare)
use_cloudflared = True  # @param {type:"boolean"}

# Basic environment tweaks for Colab
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:256"
os.environ["XDG_RUNTIME_DIR"] = "/tmp"
os.environ["SDL_AUDIODRIVER"] = "dummy"
os.environ["PYGAME_HIDE_SUPPORT_PROMPT"] = "1"
os.environ["FFMPEG_LOGLEVEL"] = "quiet"

def start_comfy():
    cmd = ["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"]
    subprocess.Popen(cmd)

# Start ComfyUI in the background
thread = threading.Thread(target=start_comfy, daemon=True)
thread.start()

# Give it a bit of time to boot
time.sleep(10)
print("ComfyUI server started on port 8188.")

public_url = None

if use_cloudflared:
    print("Using cloudflared tunnel (public-ish URL).")

    # Download standalone cloudflared binary once and make it executable
    if not os.path.exists("/usr/local/bin/cloudflared"):
        !curl -L https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
        !chmod +x /usr/local/bin/cloudflared
    else:
        print("cloudflared already present, skipping download.")

    print("Starting cloudflared tunnel... (watch below for the https URL)")
    cf_proc = subprocess.Popen(
        ["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    # Read stdout and look specifically for the trycloudflare URL
    start_time = time.time()
    try:
        while time.time() - start_time < 120:  # up to ~2 minutes to find URL
            line = cf_proc.stdout.readline()
            if not line:
                break
            line_strip = line.strip()
            print(line_strip)
            if "trycloudflare.com" in line_strip:
                m = re.search(r"https://[^\s]+", line_strip)
                if m:
                    public_url = m.group(0)
                    print(f"\nDetected tunnel URL: {public_url}")
                    break
    except Exception as e:
        print("Could not parse cloudflared output:", e)
else:
    print("Using Colab's internal proxy (no external tunnel).")
    try:
        from google.colab.output import eval_js
        public_url = eval_js("google.colab.kernel.proxyPort(8188)")
        print("Proxy URL:", public_url)
        display(HTML(f'<iframe src="{public_url}" style="width: 100%; height: 600px; border: 0;"></iframe>'))
    except Exception as e:
        print("Could not auto-create iframe / proxy URL:", e)
        print("You can still tunnel manually with cloudflared or ngrok if desired.")

if public_url:
    print(f"\nComfyUI is available at: {public_url}")
else:
    print("\nNo public URL detected yet. If cloudflared is still running, wait a few seconds and re-run this cell.")

print(
    '''
Next steps (inside ComfyUI):

1. In ComfyUI, go to the file browser and load the Lynx T2V workflow:
   custom_nodes/ComfyUI-WanVideoWrapper/example_workflows/wanvideo_T2V_14B_lynx_example_01.json

2. Make sure the following models are loaded in the workflow nodes:
   - Unet Loader (GGUF):  Wan2_1-T2V-14B-Q4_K_M.gguf
   - Text encoder:        umt5_xxl_fp8_e4m3fn_scaled.safetensors
   - VAE:                 wan_2.1_vae.safetensors
   - Lynx IP/Ref/Resampler: the files in models/lynx

3. Add or confirm the Lightx2v T2V LoRA node in the graph
   and set its strength and steps appropriate for your VRAM/time budget
   (for example, ~0.8 strength and 4 steps with an LCM sampler).

4. Create or confirm up to four LoRA loader nodes chained on the UNet
   for the LoRAs you downloaded into models/loras.
   Each LoRA loader node has its own strength slider that you can tune
   directly in the ComfyUI interface.

5. In the text prompt node(s), set your prompt and negative prompt
   however you like for the current shot.

6. In the Lynx node, set ip_scale and ref_scale to control
   how strongly the identity and reference details are enforced
   (for example ~0.9 / 0.9 as a starting point).

7. Queue the prompt. The output video will be saved under:
   /content/ComfyUI/output

You can then download the video via the Colab file browser.
'''
)

# Keep the cell alive so the tunnel and server remain accessible
print("\nKeeping this cell alive; stop it manually when you're done using ComfyUI.")
try:
    while True:
        time.sleep(300)
except KeyboardInterrupt:
    print("Stopping keep-alive loop. Cloudflared/ComfyUI will stop when the processes or runtime end.")

/content/ComfyUI
ComfyUI server started on port 8188.
Using cloudflared tunnel (public-ish URL).
cloudflared already present, skipping download.
Starting cloudflared tunnel... (watch below for the https URL)
2025-12-11T22:29:30Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2025-12-11T22:29:30Z INF Requesting new quick Tunnel on trycloudflare.com...
2025-12-11T22:29:35Z INF +---------------------------------------------------------------------------------------